In [2]:
from pynq import Overlay
import time

In [17]:
# ==========================================
# 1. 載入硬體與綁定控制器
# ==========================================
print("載入解密硬體中...")
overlay = Overlay("RFC8439.bit")

gpio_len  = overlay.axi_gpio_0
gpio_ctrl = overlay.axi_gpio_1
bram_src  = overlay.axi_bram_ctrl_0
bram_dst  = overlay.axi_bram_ctrl_1

# ==========================================
# 2. 設定參數
# ==========================================
MSG_LEN = 16768
AD_LEN = 0

gpio_len.channel1.write(MSG_LEN, 0xFFFF_FFFF)
gpio_len.channel2.write(AD_LEN, 0xFFFF_FFFF)
print(f"✅ 參數設定完成: MSG_LEN={MSG_LEN}, AD_LEN={AD_LEN}")

# ==========================================
# 3. 讀取 Src_RAM_dec.txt 並寫入 BRAM0
# ==========================================
# 檔案格式: [key][nonce][AAD][padding][ciphertext][tag]
print("📥 正在將 Src_RAM_dec.txt 寫入 BRAM0...")
with open("stress_Src_RAM_dec.txt", "r") as f:
    src_lines = f.readlines()

for i, line in enumerate(src_lines):
    data = int(line.strip(), 16) 
    bram_src.write(i * 4, data) 

print("✅ BRAM0 寫入完成！")

# ==========================================
# 4. 計算偏移量並啟動硬體 (解密模式)
# ==========================================
# 硬體內部 align16(44 + AD_LEN) 的軟體重現
offset = (44 + AD_LEN + 15) & 0xFFFFFFF0
print(f"🔍 預期密文起始位址 (Offset): {offset} Bytes")
print(f"🔍 預期 Tag 起始位址 (Offset): {offset + MSG_LEN} Bytes")

載入解密硬體中...
✅ 參數設定完成: MSG_LEN=16768, AD_LEN=0
📥 正在將 Src_RAM_dec.txt 寫入 BRAM0...
✅ BRAM0 寫入完成！
🔍 預期密文起始位址 (Offset): 48 Bytes
🔍 預期 Tag 起始位址 (Offset): 16816 Bytes


In [18]:

print("🚀 啟動 RFC8439 解密引擎...")
gpio_ctrl.channel1.write(0x2, 0x3) 
time.sleep(0.01)                    
gpio_ctrl.channel1.write(0x3, 0x3)  # 觸發運算 (start = 1)

# ==========================================
# 5. 等待運算與 Tag 認證 (mac_error 比對)
# ==========================================
print("⏳ 等待硬體解密與 Tag 認證...")
start_time = time.time()
timeout = 5.0 

while True:
    status = gpio_ctrl.channel2.read()
    if (status & 0x1) != 0:  # 檢查 bit 0 (done)
        break
    if (time.time() - start_time) > timeout:
        print("❌ 錯誤：硬體逾時 (Timeout)！")
        break

if (status & 0x1) != 0:
    print("✅ 硬體運算完成！")
    
    # 檢查 bit 1 (mac_error)
    # 這是硬體自動把 Poly1305 的結果與 Src_RAM 最後 16 Bytes 比對的結果
    mac_err_flag = (status & 0x2) >> 1
    if mac_err_flag == 1:
        print("🚨 警告：MAC 認證失敗 (Tag 不符)！密文可能遭到竄改。")
    else:
        print("🔒 MAC 認證成功！Poly1305 計算出的 Tag 與 Src_RAM 結尾的 Tag 完全相符。")

# 拉低 start 準備下一次運算
gpio_ctrl.channel1.write(0x2, 0x3)

# ==========================================
# 6. 比對 BRAM1 解密結果 (明文)
# ==========================================
print("📤 正在從 BRAM1 讀取資料並比對 Dst_RAM_dec.txt...")
try:
    with open("stress_Dst_RAM_dec.txt", "r") as f:
        dst_lines = f.readlines()

    error_count = 0
    for i, line in enumerate(dst_lines):
        expected_data = int(line.strip(), 16)
        actual_data = bram_dst.read(i * 4)
        
        if actual_data != expected_data:
            if error_count < 10:
                print(f"❌ 錯誤 @ Word {i} (Offset 0x{i*4:04x}): 預期 0x{expected_data:08x}, 實際出 0x{actual_data:08x}")
            error_count += 1

    if error_count == 0:
        print("🎉 恭喜！資料比對完全正確，硬體解密驗證成功！")
    else:
        print(f"💔 驗證失敗：總共有 {error_count} 個 Word 與預期不符。")
    gpio_counter = overlay.axi_gpio_2
    hw_cycles = gpio_counter.channel1.read()

# 4. 奈秒級轉換 (100MHz = 10ns per cycle)
    CLOCK_PERIOD_NS = 25 
    hardware_time_ns = hw_cycles * CLOCK_PERIOD_NS
    hardware_time_us = hardware_time_ns / 1000

    print(f"⏱️ 硬體消耗週期: {hw_cycles} cycles")
    print(f"🚀 精準硬體處理時間: {hardware_time_ns} 奈秒 ({hardware_time_us:.2f} 微秒)")

# 5. 計算真實吞吐量 (Throughput)
    throughput_MBps = (MSG_LEN / 1024 / 1024) / (hardware_time_ns * 1e-9)
    print(f"⚡ 硬體吞吐量: {throughput_MBps:.2f} MB/s")
except FileNotFoundError:
    print("⚠️ 找不到 Dst_RAM_dec.txt 檔案，略過明文比對。")


🚀 啟動 RFC8439 解密引擎...
⏳ 等待硬體解密與 Tag 認證...
✅ 硬體運算完成！
🔒 MAC 認證成功！Poly1305 計算出的 Tag 與 Src_RAM 結尾的 Tag 完全相符。
📤 正在從 BRAM1 讀取資料並比對 Dst_RAM_dec.txt...
🎉 恭喜！資料比對完全正確，硬體解密驗證成功！
⏱️ 硬體消耗週期: 21062 cycles
🚀 精準硬體處理時間: 526550 奈秒 (526.55 微秒)
⚡ 硬體吞吐量: 30.37 MB/s
